# Train and Validate
* Train a model across the specified training sites and sampling approach
* Validate the model at the excluded site

## Todo
* Pull out probabilities using model.predict_proba(observations_to_predict)

In [ ]:
import pathlib
import numpy
import dask.distributed
import pandas
import joblib

module_path = pathlib.Path.cwd().parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling

%load_ext autoreload
%autoreload 2

# Values to edit

In [ ]:
sample_method = "sampling_2"
method_2_threshold = .98 # .98 .99 .97 .95

In [ ]:
# View the amount of training data for the selected sampling apporach
samples_per_site = pandas.read_csv(utils.get_samples_summary_file_path(sample_method, method_2_threshold))
samples_per_site

In [ ]:
all_training_sites =  ["CatlinsRiverMouth", "CatlinsLake", "Duvauchelle",
                       "Robinsons", "Childrens", "Takamatua", "Ihutai", "Purau"] 

In [ ]:
test_site = "Ihutai"
training_sites = ["Duvauchelle", "Robinsons", "Childrens", "Takamatua"] # all_training_sites.copy() # ["Duvauchelle", "Ihutai", "Purau"]
model_file = utils.get_model_file(f"trained_on_akaroa_sites_tested_on_{test_site}")

prediction_file = data_path / "validation" / "predictions" / f"{test_site}_prediction_{model_file.stem}.nc"

# Ensure not training on test site
if test_site in training_sites:
    training_sites.remove(test_site)


In [ ]:
# Mappings of Satellite classes to consider from UAV classes
uav_classes_to_ignore = ['Shadow', 'Glare']
satellite_classes = {'Seagrass': 1, 'Mixed': 2, 'Unvegetated': 3, 'Water': 4,}
satellite_from_uav_classes = {'Seagrass': ['Seagrass', 'Seagrass submerged'],
 'Mixed': ['Gracilaria', 'Ulva', 'Cystophora', 'Hormosira', 'Gracilaria submerged', 'Submerged vegetation',
           'Brown algae mixed', 'Microphytobenthos', 'Green algae mixed', 
           'Filamentous brown algae', 'Ulva mats', 'Terrestrial', 'Red algae mixed'],
  'Unvegetated': ['Saltmarsh', 'Unvegetated', 'Rock', ],
                              
  'Water': ['Water'],
}

# Cells to run
* Train and save model
* Review model
  * satellite bands of each UAV class
  * satellite bands of each satellite class
  * importance of satellite bands in trained model
* Validation
  * Predict excluded site
  * Plot confusion matrix comparing prediction to UAV classifications

In [ ]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

In [ ]:
data_path = utils.get_data_path()
utils.create_data_folders()
uav_labels_file = data_path / "ELF24505_ClassificationClasses.txt"
sample_folder = utils.get_samples_path(sample_method=sample_method, method_2_threshold=method_2_threshold)

test_satellite_file = data_path / "training" / "satellite_images" / f"{test_site}_sentinel-2.nc"
test_polygon_file = polygon_file = utils.get_site_polygon_path(test_site)
test_uav_file = data_path / "classified_orthos" / f"{test_site}_classified.tif"

### Train and save model

In [ ]:
# train and save the model
model, training_dataframe = training.train_classifier(
    training_sites=training_sites,
    samples_path=sample_folder,
    uav_labels_file=uav_labels_file,
    uav_classes_to_ignore=uav_classes_to_ignore,
    satellite_classes=satellite_classes,
    satellite_from_uav_classes=satellite_from_uav_classes)

# Save model
joblib.dump(model, model_file); # , compress=3);

# Save a record of the classes considered in training
satellite_classes_dataframe = pandas.DataFrame.from_dict(satellite_classes, orient='index', columns=['satellite_class_id'])
satellite_classes_dataframe['uav_class_ids'] = satellite_classes_dataframe.index.map(lambda key: f"{satellite_from_uav_classes[key]}")
satellite_classes_dataframe.to_csv(model_file.with_name(f"{model_file.stem}_class_mappings.csv"), index=True)

print("Satellite training classes present: "
      f"{[key for key, value in satellite_classes.items() if value in training_dataframe['satellite_class_id'].unique()]}")

### Review model

In [ ]:
training.plot_training_data_class_distribution(training_dataframe=training_dataframe, model_file=model_file)

In [ ]:
training.plot_model_feature_importance(training_dataframe=training_dataframe, model_file=model_file)

### Validate 

In [ ]:
predictions, satellite_data = training.predict_site(test_satellite_file=test_satellite_file, polygon_file=test_polygon_file, model_file=model_file)
utils.save_netcdf(predictions, prediction_file)

print(
    f"Satellite training classes present: {[key for key, value in satellite_classes.items() if value in training_dataframe['satellite_class_id'].unique()]}. "
    f"Predicted classes present: {[key for key, value in satellite_classes.items() if value in numpy.unique(predictions.data)]}"
     )

In [ ]:
training.confusion_matrix_of_site(
    test_uav_file=test_uav_file,
    uav_labels_file=uav_labels_file,
    prediction_file=prediction_file,
    satellite_classes=satellite_classes,
    satellite_from_uav_classes=satellite_from_uav_classes,
    uav_classes_to_ignore=uav_classes_to_ignore,
    polygon_file=test_polygon_file)